In [1]:
import pandas as pd
import numpy as np
import mlflow
import dagshub

dagshub.init(repo_name = 'yt_chrome_plugin', repo_owner = 'AmanGupta8423', mlflow = True)
mlflow.set_experiment("Experiment3-handling-imbalanced-data")
mlflow.set_tracking_uri('https://dagshub.com/AmanGupta8423/yt_chrome_plugin.mlflow')

c:\Users\win11\OneDrive\Desktop\MLops\Project\yt_plugin\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Accessing as AmanGupta8423

Initialized MLflow to track repo "AmanGupta8423/yt_chrome_plugin"

Repository AmanGupta8423/yt_chrome_plugin initialized!

2026/11/26 19:07:18 INFO mlflow.tracking.fluent: Experiment with name 'Experiment3-handling-imbalanced-data' does not exist. Creating a new experiment.


In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix
from sklearn.model_selection import train_test_split
import mlflow.sklearn
import matplotlib.pyplot as plt
import seaborn as sns
import os
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler,SMOTE,ADASYN
from imblearn.combine import SMOTEENN

In [3]:
df = pd.read_csv('./data/raw/dataset.csv').dropna(subset='clean_comment')
df.shape

(36662, 2)

In [4]:
def run_imbalanced_experiment(imbalanced_method):
    vectorizer = TfidfVectorizer(max_features=3000, ngram_range=(1,3))

    X_train,X_test,y_train,y_test = train_test_split(df['clean_comment'],df['category'],test_size=0.2,random_state=42,stratify=df['category'])
    X_train = vectorizer.fit_transform(X_train)
    X_test = vectorizer.transform(X_test)

    if imbalanced_method == 'class_weight':
        class_weight = 'balanced'
        X_train_balanced = X_train
        y_train_balanced = y_train
    else:
        class_weight = None

        if imbalanced_method == 'oversampling':
            ros = RandomOverSampler(random_state=42)
            X_train_balanced,y_train_balanced = ros.fit_resample(X_train,y_train)
        elif imbalanced_method == 'smote':
            smote = SMOTE(random_state= 42)
            X_train_balanced,y_train_balanced = smote.fit_resample(X_train,y_train)
        elif imbalanced_method == 'adasyn':
            adasyn = ADASYN(random_state=42)
            X_train_balanced,y_train_balanced = adasyn.fit_resample(X_train,y_train)
        elif imbalanced_method == 'undersampling':
            rus = RandomUnderSampler(random_state=42)
            X_train_balanced,y_train_balanced = rus.fit_resample(X_train,y_train)
        elif imbalanced_method == 'smote_enn':
            smote_enn = SMOTEENN(random_state=42)
            X_train_balanced,y_train_balanced = smote_enn.fit_resample(X_train,y_train)

    with mlflow.start_run() as run:
        mlflow.set_tag('mlflow.runName', f'Tfidf_trigram_max_features:3000_imbalanced_method: {imbalanced_method}_RandomForest')
        mlflow.set_tag('experiment_type','feature_engineering')
        mlflow.set_tag('model_type', 'RandomForestClassifier')
        mlflow.set_tag('description','Finding the best Imbalaced data handling technique')

        mlflow.log_param('vectorizer_name', 'TF-IDF')
        mlflow.log_param('ngrams_range','Trigram')   
        mlflow.log_param('max_features',3000)  
        mlflow.log_param('imbalanced_method',imbalanced_method )

        n_estimators = 100
        max_depth = 10
        mlflow.log_param('n_estimators',n_estimators)
        mlflow.log_param('max_depth',max_depth)
        mlflow.log_param('class_weight',class_weight)

        model = RandomForestClassifier(n_estimators=n_estimators,max_depth=max_depth, random_state= 42, class_weight=class_weight,n_jobs=-1)

        model.fit(X_train_balanced,y_train_balanced)
        y_pred = model.predict(X_test)

        accuracy = accuracy_score(y_test,y_pred)

        classification_rep = classification_report(y_test,y_pred,output_dict=True)
            
        for label,metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f'{label}_{metric}',value)

        conf_matrix = confusion_matrix(y_test,y_pred)
        plt.figure(figsize=(8,6))
        sns.heatmap(conf_matrix,annot = True, fmt = 'd',cmap='Blues')
        plt.xlabel('Predicted')
        plt.ylabel('Actual')
        plt.title(f'Confusion Matrix: RandomForest with Tfidf Trigram max_features:3000 and Imabalanced technique: {imbalanced_method}')
    
        os.makedirs("./reports", exist_ok=True)

        plt.savefig('./reports/confusion_matrix.png')
        mlflow.log_artifact('./reports/confusion_matrix.png')
        plt.close()

        mlflow.sklearn.log_model(model, name="random_forest_model",skops_trusted_types=["sklearn.tree._tree.Tree"])
        print(accuracy)

imbalanced_methods = ['class_weight','undersampling','oversampling','smote','adasyn','smote_enn']
for imbalanced_method in imbalanced_methods:
    run_imbalanced_experiment(imbalanced_method)

0.6627573980635484
🏃 View run Tfidf_trigram_max_features:3000_imbalanced_method: class_weight_RandomForest at: https://dagshub.com/AmanGupta8423/yt_chrome_plugin.mlflow/#/experiments/4/runs/4433d84c3958496e94965b17b0b83dc1
🧪 View experiment at: https://dagshub.com/AmanGupta8423/yt_chrome_plugin.mlflow/#/experiments/4
0.6679394517932633
🏃 View run Tfidf_trigram_max_features:3000_imbalanced_method: undersampling_RandomForest at: https://dagshub.com/AmanGupta8423/yt_chrome_plugin.mlflow/#/experiments/4/runs/8be8320d8a964f6db16ca47959781c7a
🧪 View experiment at: https://dagshub.com/AmanGupta8423/yt_chrome_plugin.mlflow/#/experiments/4
0.6673939724532933
🏃 View run Tfidf_trigram_max_features:3000_imbalanced_method: oversampling_RandomForest at: https://dagshub.com/AmanGupta8423/yt_chrome_plugin.mlflow/#/experiments/4/runs/3cca3c71181d452abae5e19d1196bf03
🧪 View experiment at: https://dagshub.com/AmanGupta8423/yt_chrome_plugin.mlflow/#/experiments/4
0.6712123278330833
🏃 View run Tfidf_trigra

In [ ]:
# SMOTE or Under Sampling dono shi perform kiye